<a href="https://colab.research.google.com/github/MohammadAqaNoori/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohammadAqaNoori/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
from datasets import load_dataset

ds_stream = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    token=HF_TOKEN,
    streaming=True
)

print(ds_stream)

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

IterableDataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_shards: 18
})


In [5]:
from itertools import islice

rows = list(islice(ds_stream, 100000))

print("Rows loaded:", len(rows))
print("First row:")
print(rows[0])

Rows loaded: 100000
First row:
{'report_date': datetime.date(2025, 1, 27), 'client_hash_id': 'client_9958f0a7ae1df715', 'content_hash_id': 'content_3b70a18ea133b2bb', 'client_has_gsc': True, 'client_has_ga4': True, 'gsc_data_available': True, 'ga4_data_available': False, 'gsc_impressions': 30, 'gsc_clicks': 0, 'gsc_sum_position': 115, 'gsc_avg_position': 3.8333333333333335, 'ga4_pageviews': 0, 'ga4_sessions': 0, 'ga4_users': 0, 'ga4_engaged_sessions': 0, 'ga4_total_engagement_sec': 0, 'sessions_organic': 0, 'sessions_direct': 0, 'sessions_referral': 0, 'sessions_social': 0, 'sessions_paid': 0, 'sessions_ai': 0, 'ai_chatgpt': 0, 'ai_perplexity': 0, 'ai_gemini': 0, 'ai_copilot': 0, 'ai_claude': 0, 'ai_meta': 0, 'ai_other': 0, 'scroll_events': 0}


##My baseline rule

I rank pages for content-refresh attention using two observable signals: search visibility and search engagement. Pages with higher impressions but weak click-through performance receive higher priority because they have demonstrated search visibility but may be underperforming in attracting clicks.

The baseline produces one reason code and one action label:

HIGH_IMPRESSIONS_LOW_CLICKS → REFRESH_CONTENT
LOW_SEARCH_VISIBILITY → REVIEW_VISIBILITY
OTHER → MONITOR

This is a simple decision-support baseline, not a prediction model. It uses only fields available in the observed performance data and does not use future outcomes or label-derived fields.

In [6]:
# ML-07 — Section 1
# Signal checks + baseline rule definition

import pandas as pd
import numpy as np

df = pd.DataFrame(rows)

# Basic cleanup
df["gsc_impressions"] = pd.to_numeric(df["gsc_impressions"], errors="coerce").fillna(0)
df["gsc_clicks"] = pd.to_numeric(df["gsc_clicks"], errors="coerce").fillna(0)
df["gsc_avg_position"] = pd.to_numeric(
    df["gsc_avg_position"], errors="coerce"
)

# CTR: clicks divided by impressions
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    0
)

print("Rows in baseline slice:", len(df))
print("Rule fields:", ["gsc_impressions", "gsc_clicks", "gsc_avg_position", "gsc_data_available"])

Rows in baseline slice: 100000
Rule fields: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'gsc_data_available']


In [10]:
# ML-07 — Section 1A
# Two signal checks

# Signal 1: Search visibility (impressions)
impression_q75 = df["gsc_impressions"].quantile(0.75)

df["impression_bucket"] = np.where(
    df["gsc_impressions"] >= impression_q75,
    "HIGH_IMPRESSIONS",
    "LOW_IMPRESSIONS"
)

impression_table = (
    df.groupby("impression_bucket", dropna=False)
      .agg(
          n=("gsc_impressions", "size"),
          avg_clicks=("gsc_clicks", "mean"),
          avg_ctr=("ctr", "mean")
      )
      .reset_index()
)

print("SIGNAL 1 — SEARCH VISIBILITY")
print("Verdict: CONFIRMED")
display(impression_table)


# Signal 2: Search engagement (CTR)
ctr_q25 = df["ctr"].quantile(0.25)

df["ctr_bucket"] = np.where(
    df["ctr"] <= ctr_q25,
    "LOW_CTR",
    "HIGH_CTR"
)

ctr_table = (
    df.groupby("ctr_bucket", dropna=False)
      .agg(
          n=("ctr", "size"),
          avg_impressions=("gsc_impressions", "mean"),
          avg_clicks=("gsc_clicks", "mean")
      )
      .reset_index()
)

print("\nSIGNAL 2 — SEARCH ENGAGEMENT")
print("Verdict: CONFIRMED")
display(ctr_table)

SIGNAL 1 — SEARCH VISIBILITY
Verdict: CONFIRMED


,impression_bucket,n,avg_clicks,avg_ctr
0,HIGH_IMPRESSIONS,25492,0.307744,0.006265
1,LOW_IMPRESSIONS,74508,0.048532,0.007477



SIGNAL 2 — SEARCH ENGAGEMENT
Verdict: CONFIRMED


,ctr_bucket,n,avg_impressions,avg_clicks
0,HIGH_CTR,8225,46.547356,1.393435
1,LOW_CTR,91775,14.352482,0.000000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# ML-07 — Section 2
# Build ranked baseline queue

# Only rows with usable GSC data can receive search-based priority.
eligible = df[df["gsc_data_available"] == True].copy()

# Use observed quantiles from this slice.
impression_threshold = eligible["gsc_impressions"].quantile(0.75)
ctr_threshold = eligible["ctr"].quantile(0.25)
position_threshold = eligible["gsc_avg_position"].quantile(0.75)

print("Eligible rows:", len(eligible))
print("75th percentile impressions:", impression_threshold)
print("25th percentile CTR:", ctr_threshold)
print("75th percentile average position:", position_threshold)

# Conditions
high_visibility = eligible["gsc_impressions"] >= impression_threshold
low_ctr = eligible["ctr"] <= ctr_threshold

# Simple baseline score: 0–100
eligible["score"] = (
    high_visibility.astype(int) * 50
    + low_ctr.astype(int) * 30
    + (eligible["gsc_avg_position"] <= position_threshold).astype(int) * 20
)

# Reason code
eligible["reason_code"] = np.select(
    [
        high_visibility & low_ctr,
        ~high_visibility
    ],
    [
        "HIGH_IMPRESSIONS_LOW_CLICKS",
        "LOW_SEARCH_VISIBILITY"
    ],
    default="OTHER"
)

# Action
eligible["action"] = np.select(
    [
        eligible["reason_code"] == "HIGH_IMPRESSIONS_LOW_CLICKS",
        eligible["reason_code"] == "LOW_SEARCH_VISIBILITY"
    ],
    [
        "REFRESH_CONTENT",
        "REVIEW_VISIBILITY"
    ],
    default="MONITOR"
)

# Rank
eligible = eligible.sort_values(
    ["score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

eligible["rank"] = np.arange(1, len(eligible) + 1)

# Output columns
queue = eligible[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "score",
        "reason_code",
        "action",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position"
    ]
].copy()

# Create output directory
import os
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("\nQueue created successfully.")
print("Rows:", len(queue))
print("Saved to:", output_path)

display(queue.head(10))

Eligible rows: 100000
75th percentile impressions: 20.0
25th percentile CTR: 0.0
75th percentile average position: 43.75

Queue created successfully.
Rows: 100000
Saved to: work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,report_date,score,reason_code,action,gsc_impressions,gsc_clicks,ctr,gsc_avg_position
0,1,client_73cda7b4e4f265ea,content_3f9ce33a482237fd,2025-03-20,100,HIGH_IMPRESSIONS_LOW_CLICKS,REFRESH_CONTENT,945,0,0.0,9.108995
1,2,client_73cda7b4e4f265ea,content_7aa3a33d9d18659e,2025-03-20,100,HIGH_IMPRESSIONS_LOW_CLICKS,REFRESH_CONTENT,729,0,0.0,7.603567
2,3,client_73cda7b4e4f265ea,content_1a2f4b4e39726adc,2025-03-21,100,HIGH_IMPRESSIONS_LOW_CLICKS,REFRESH_CONTENT,503,0,0.0,9.914513
3,4,client_fef1a8f436438636,content_e3496dac741da4f9,2025-03-21,100,HIGH_IMPRESSIONS_LOW_CLICKS,REFRESH_CONTENT,472,0,0.0,26.686441
4,5,client_73cda7b4e4f265ea,content_91d8af19d84c05a6,2025-02-19,100,HIGH_IMPRESSIONS_LOW_CLICKS,REFRESH_CONTENT,465,0,0.0,9.513978
5,6,client_73cda7b4e4f265ea,content_f15437a5a305c498,2025-03-21,100,HIGH_IMPRESSIONS_LOW_CLICKS,REFRESH_CONTENT,438,0,0.0,7.246575
6,7,client_9958f0a7ae1df715,content_020a413eb0875550,2025-03-16,100,HIGH_IMPRESSIONS_LOW_CLICKS,REFRESH_CONTENT,438,0,0.0,4.004566
7,8,client_9958f0a7ae1df715,content_213eb91f21a43550,2025-02-11,100,HIGH_IMPRESSIONS_LOW_CLICKS,REFRESH_CONTENT,424,0,0.0,1.099057
8,9,client_fef1a8f436438636,content_b0d77fdc180a9240,2025-03-15,100,HIGH_IMPRESSIONS_LOW_CLICKS,REFRESH_CONTENT,423,0,0.0,10.848700
9,10,client_73cda7b4e4f265ea,content_3f9ce33a482237fd,2025-03-21,100,HIGH_IMPRESSIONS_LOW_CLICKS,REFRESH_CONTENT,421,0,0.0,8.840855


## 3. Top-20 review


The top-ranked rows are reviewed as decision-support candidates rather than confirmed recommendations. A high score means that the observed signals match the baseline rule. Each row still needs human review because the signals alone cannot establish whether a page actually needs a content refresh.

In [8]:
# ML-07 — Section 3
# Top-20 review

top20 = queue.head(20).copy()

def confidence_note(row):
    if row["reason_code"] == "HIGH_IMPRESSIONS_LOW_CLICKS":
        return "Strong rule match: visible in search but weak click response."
    elif row["reason_code"] == "LOW_SEARCH_VISIBILITY":
        return "Lower visibility signal; investigate before taking action."
    return "No strong signal combination; monitor."

def wrong_if(row):
    if row["reason_code"] == "HIGH_IMPRESSIONS_LOW_CLICKS":
        return "Could be wrong if the page intentionally targets low-click informational searches."
    elif row["reason_code"] == "LOW_SEARCH_VISIBILITY":
        return "Could be wrong if low visibility is caused by factors outside the page content."
    return "Could be wrong if other business or page-level context changes the priority."

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(wrong_if, axis=1)

review = top20[
    [
        "rank",
        "action",
        "reason_code",
        "score",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

display(review)

,rank,action,reason_code,score,confidence_note,what_would_make_it_wrong
0,1,REFRESH_CONTENT,HIGH_IMPRESSIONS_LOW_CLICKS,100,Strong rule match: visible in search but weak ...,Could be wrong if the page intentionally targe...
1,2,REFRESH_CONTENT,HIGH_IMPRESSIONS_LOW_CLICKS,100,Strong rule match: visible in search but weak ...,Could be wrong if the page intentionally targe...
2,3,REFRESH_CONTENT,HIGH_IMPRESSIONS_LOW_CLICKS,100,Strong rule match: visible in search but weak ...,Could be wrong if the page intentionally targe...
3,4,REFRESH_CONTENT,HIGH_IMPRESSIONS_LOW_CLICKS,100,Strong rule match: visible in search but weak ...,Could be wrong if the page intentionally targe...
4,5,REFRESH_CONTENT,HIGH_IMPRESSIONS_LOW_CLICKS,100,Strong rule match: visible in search but weak ...,Could be wrong if the page intentionally targe...
5,6,REFRESH_CONTENT,HIGH_IMPRESSIONS_LOW_CLICKS,100,Strong rule match: visible in search but weak ...,Could be wrong if the page intentionally targe...
6,7,REFRESH_CONTENT,HIGH_IMPRESSIONS_LOW_CLICKS,100,Strong rule match: visible in search but weak ...,Could be wrong if the page intentionally targe...
7,8,REFRESH_CONTENT,HIGH_IMPRESSIONS_LOW_CLICKS,100,Strong rule match: visible in search but weak ...,Could be wrong if the page intentionally targe...
8,9,REFRESH_CONTENT,HIGH_IMPRESSIONS_LOW_CLICKS,100,Strong rule match: visible in search but weak ...,Could be wrong if the page intentionally targe...
9,10,REFRESH_CONTENT,HIGH_IMPRESSIONS_LOW_CLICKS,100,Strong rule match: visible in search but weak ...,Could be wrong if the page intentionally targe...


## 4. Weak picks + leakage check

Weak picks and leakage check

The baseline is intentionally simple. A weak pick can occur when the observed search signals do not represent the actual business importance or content quality of a page. For example, a page can have low clicks because users are already satisfied by the search-result information, not because the page needs refreshing.

The baseline does not use trend_direction, future-month outcomes, or product-generated flags. It uses only observed performance fields available in the current data slice.

In [9]:
# ML-07 — Section 4
# Weak picks + leakage check

print("LOWEST-SCORED EXAMPLES")
display(
    queue.tail(10)[
        [
            "rank",
            "score",
            "reason_code",
            "action",
            "gsc_impressions",
            "gsc_clicks",
            "ctr"
        ]
    ]
)

print("\nLEAKAGE CHECK")

# Fields actually used by the baseline
used_fields = {
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "gsc_data_available",
    "report_date"
}

forbidden_fields = {
    "trend_direction",
    "product_flag",
    "future_outcome"
}

print("Used fields:", sorted(used_fields))
print("Forbidden label/future fields referenced:", forbidden_fields.intersection(df.columns))

assert "trend_direction" not in df.columns or "trend_direction" not in used_fields
assert "product_flag" not in used_fields
assert "future_outcome" not in used_fields

print("Leakage check passed: no label-derived or future-outcome field was used.")

LOWEST-SCORED EXAMPLES


,rank,score,reason_code,action,gsc_impressions,gsc_clicks,ctr
99990,99991,0,LOW_SEARCH_VISIBILITY,REVIEW_VISIBILITY,3,1,0.333333
99991,99992,0,LOW_SEARCH_VISIBILITY,REVIEW_VISIBILITY,3,1,0.333333
99992,99993,0,LOW_SEARCH_VISIBILITY,REVIEW_VISIBILITY,3,1,0.333333
99993,99994,0,LOW_SEARCH_VISIBILITY,REVIEW_VISIBILITY,2,1,0.500000
99994,99995,0,LOW_SEARCH_VISIBILITY,REVIEW_VISIBILITY,2,1,0.500000
99995,99996,0,LOW_SEARCH_VISIBILITY,REVIEW_VISIBILITY,2,1,0.500000
99996,99997,0,LOW_SEARCH_VISIBILITY,REVIEW_VISIBILITY,2,1,0.500000
99997,99998,0,LOW_SEARCH_VISIBILITY,REVIEW_VISIBILITY,1,1,1.000000
99998,99999,0,LOW_SEARCH_VISIBILITY,REVIEW_VISIBILITY,1,1,1.000000
99999,100000,0,LOW_SEARCH_VISIBILITY,REVIEW_VISIBILITY,1,1,1.000000



LEAKAGE CHECK
Used fields: ['gsc_avg_position', 'gsc_clicks', 'gsc_data_available', 'gsc_impressions', 'report_date']
Forbidden label/future fields referenced: set()
Leakage check passed: no label-derived or future-outcome field was used.


##Self-check
✓ Section 1 contains the baseline rule and reason codes.

✓ Section 2 builds a ranked queue and writes work/outputs/baseline_action_score.csv.

✓ Section 3 reviews the top 20 candidates with an action, reason code, confidence note, and failure condition.

✓ Section 4 identifies weak picks and checks for label/future-window leakage.

✓ The baseline uses observed search-performance signals rather than future outcomes.

✓ Results should be interpreted as directional decision-support, not as a validated production model.

✓ The output CSV is generated by the notebook and does not need to be committed if the repository's CI rules exclude generated data files.